# Gradient Boosting Parcel Volume Forecasting

This notebook tests Gradient Boosting against the existing baseline, Linear Regression and Random Forest models using the same time-based test period.

> The dataset is synthetic and contains no confidential DHL data.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

DATA_PATH = Path("../data/dhl_style_parcel_volume.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["date"])

## Recreate features and the time-based split

In [ ]:
df = df.sort_values(["depot", "date"]).copy()
df["lag_1"] = df.groupby("depot")["parcel_volume"].shift(1)
df["lag_7"] = df.groupby("depot")["parcel_volume"].shift(7)
df["month"] = df["date"].dt.month
df["day_of_week_num"] = df["date"].dt.dayofweek

model_df = df.dropna(subset=["lag_1", "lag_7"]).copy()

features = [
    "lag_1",
    "lag_7",
    "day_of_week_num",
    "month",
    "is_bank_holiday",
    "is_black_friday_period",
    "is_christmas_peak",
]

split_date = "2026-01-01"
train_df = model_df[model_df["date"] < split_date].copy()
test_df = model_df[model_df["date"] >= split_date].copy()

X_train = train_df[features]
y_train = train_df["parcel_volume"]
X_test = test_df[features]
y_test = test_df["parcel_volume"]

## Reference model scores

In [ ]:
baseline_mae = mean_absolute_error(test_df["parcel_volume"], test_df["lag_7"])

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_mae = mean_absolute_error(y_test, linear_model.predict(X_test))

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_mae = mean_absolute_error(y_test, rf_model.predict(X_test))

## Train Gradient Boosting

Gradient Boosting builds trees sequentially. Each stage focuses on reducing the errors left by the previous stages.

In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
)

gb_model.fit(X_train, y_train)
gb_predictions = gb_model.predict(X_test)
gb_mae = mean_absolute_error(y_test, gb_predictions)

results = pd.DataFrame({
    "Model": [
        "7-day lag baseline",
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting",
    ],
    "Test MAE": [baseline_mae, linear_mae, rf_mae, gb_mae],
})

results.sort_values("Test MAE")

In [ ]:
gb_vs_baseline = ((baseline_mae - gb_mae) / baseline_mae) * 100
gb_vs_rf = ((rf_mae - gb_mae) / rf_mae) * 100

print(f"Gradient Boosting MAE: {gb_mae:.2f}")
print(f"Improvement vs baseline: {gb_vs_baseline:.1f}%")
print(f"Improvement vs Random Forest: {gb_vs_rf:.1f}%")

## Feature importance

In [ ]:
gb_feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": gb_model.feature_importances_,
}).sort_values(by="Importance", ascending=False)

gb_feature_importance

### Interpretation

The fitted Gradient Boosting model again relies most heavily on `lag_7`, at roughly **74.2%** importance. Bank-holiday status and day of week are the next strongest features.

### Current model ranking

- Baseline MAE: approximately **1,182.84**
- Linear Regression MAE: approximately **1,029.12**
- Random Forest MAE: approximately **903.34**
- Gradient Boosting MAE: approximately **854.90**

Gradient Boosting is currently the best-performing model. The next stage is to improve the feature set with additional lags and rolling statistics before further model tuning.